In [1]:
import joblib
from sentence_transformers import SentenceTransformer
import tkinter as tk
from tkinter import messagebox,ttk




In [2]:
import data_model 
from data_model import predict_severity,predict_user_text
from expert_system import run_expert_system

In [3]:
class PHQ9Scorer:
    questions = [
        "Little interest or pleasure in doing things",
        "Feeling down, depressed, or hopeless",
        "Trouble falling/staying asleep, or sleeping too much",
        "Feeling tired or having little energy",
        "Poor appetite or overeating",
        "Feeling bad about yourself — or that you are a failure or have let yourself or your family down",
        "Trouble concentrating on things",
        "Moving or speaking so slowly that other people could have noticed? Or being fidgety or restless?",
        "Thoughts that you would be better off dead or of hurting yourself in some way"
    ]

    @staticmethod
    def score(selections):
        predict_severity(selections)
        


In [4]:
class MentalHealthPredictor:
    def __init__(self):
        self.model = joblib.load(r"D:\uni\AI\aiLab\mental health project\text_random_forest_model.pkl")
        self.sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

    def predict(self, text):
        predicted_status,fuzzy=predict_user_text(text)
       
        return predicted_status, fuzzy


In [ ]:



class MentalHealthApp:
    def __init__(self, root):
        self.root = root
        self.root.title("NEURO NEST")
        self.root.geometry("800x700")
        self.root.configure(bg="#fdf6f0")

        # Colors
        self.PASTEL_GREEN = "#a8e6cf"
        self.PASTEL_BLUE = "#d0e6f7"
        self.PASTEL_PURPLE = "#e4c1f9"
        self.PASTEL_YELLOW = "#fffacc"
        self.TEXT_COLOR = "#10031f"
        self.BACKGROUND_COLOR = "#af96cc"

        self.scorer = PHQ9Scorer()
        self.predictor = MentalHealthPredictor()

        self.dashboard_frame = tk.Frame(root, bg=self.BACKGROUND_COLOR)
        self.phq_frame = tk.Frame(root, bg=self.BACKGROUND_COLOR)
        self.journal_frame = tk.Frame(root, bg=self.BACKGROUND_COLOR)

        self.build_dashboard()
        self.build_phq9_screen()
        self.build_journal_screen()

        self.show_dashboard()

    def show_dashboard(self):
        self.clear_frames()
        self.dashboard_frame.pack(fill="both", expand=True)

    def show_phq9(self):
        self.clear_frames()
        self.phq_frame.pack(fill="both", expand=True)

    def show_journal(self):
        self.clear_frames()
        self.journal_frame.pack(fill="both", expand=True)

    def clear_frames(self):
        for frame in [self.dashboard_frame, self.phq_frame, self.journal_frame]:
            frame.pack_forget()

    def build_dashboard(self):
        title = tk.Label(self.dashboard_frame, text="🧠 Mental Health Assistant",
                         font=("Helvetica", 24, "bold"), bg=self.BACKGROUND_COLOR, fg=self.TEXT_COLOR)
        title.pack(pady=40)

        phq_btn = tk.Button(self.dashboard_frame, text="PHQ-9 Depression Screening", font=("Arial", 16),
                            command=self.show_phq9, bg=self.PASTEL_GREEN, fg=self.TEXT_COLOR, width=30, height=2)
        phq_btn.pack(pady=20)

        journal_btn = tk.Button(self.dashboard_frame, text="Journal-Based Mental Health Detection", font=("Arial", 16),
                                command=self.show_journal, bg=self.PASTEL_BLUE, fg=self.TEXT_COLOR, width=30, height=2)
        journal_btn.pack(pady=20)

    def build_phq9_screen(self):
        tk.Label(self.phq_frame, text="PHQ-9 Depression Screening", font=("Arial", 20, "bold"),
                 bg=self.BACKGROUND_COLOR, fg=self.TEXT_COLOR).pack(pady=10)

        instruction = tk.Label(self.phq_frame,
            text="Answer each (0=Not at all, 1=Several days, 2=More than half the days, 3=Nearly every day)",
            font=("Arial", 11), fg=self.TEXT_COLOR, bg=self.BACKGROUND_COLOR)
        instruction.pack(pady=5)

        self.entries = []
        for i, question in enumerate(self.scorer.questions, 1):
            frame = tk.Frame(self.phq_frame, bg=self.BACKGROUND_COLOR)
            frame.pack(anchor="w", padx=20, pady=4, fill="x")

            tk.Label(frame, text=f"{i}. {question}", wraplength=600, justify="left",
                     bg=self.BACKGROUND_COLOR, fg=self.TEXT_COLOR).pack(side="left", fill="x", expand=True)
            entry = tk.Entry(frame, width=4)
            entry.pack(side="right", padx=10)
            self.entries.append(entry)

        submit_btn = tk.Button(self.phq_frame, text="Submit PHQ-9", command=self.submit_phq,
                               bg=self.PASTEL_GREEN, fg=self.TEXT_COLOR, font=("Arial", 12))
        submit_btn.pack(pady=20)

        back_btn = tk.Button(self.phq_frame, text="← Back to Dashboard", command=self.show_dashboard,
                             font=("Arial", 10), bg=self.PASTEL_YELLOW)
        back_btn.pack(pady=5)

        self.phq_result_label = tk.Label(self.phq_frame, text="", font=("Arial", 14, "bold"),
                                         bg=self.BACKGROUND_COLOR, fg=self.TEXT_COLOR)
        self.phq_result_label.pack(pady=10)

    def submit_phq(self):
        try:
            scores = []
            for entry in self.entries:
                value = entry.get().strip()
                if value == '':
                    raise ValueError("Please answer all PHQ-9 questions.")
                score = int(value)
                if score < 0 or score > 3:
                    raise ValueError("PHQ-9 scores must be between 0 and 3.")
                scores.append(score)
    
            if len(scores) != 9:
                raise ValueError("PHQ-9 must have exactly 9 answers.")
    
            severity = predict_severity(scores)
            self.phq_result_label.config(text=f"Depression Severity: {severity}")
    
            if severity in ['Moderate', 'Moderately Severe', 'Severe']:
                response = messagebox.askyesno("Start Breathing Exercise?",
                                               f"Your result shows '{severity}' depression.\n"
                                               "Would you like to try a calming breathing exercise?")
                if response:
                    self.start_breathing_exercise()
    
            for entry in self.entries:
                entry.delete(0, tk.END)
    
        except ValueError as ve:
            messagebox.showerror("Input Error", str(ve))
        except Exception as e:
            messagebox.showerror("Unexpected Error", f"An unexpected error occurred:\n{str(e)}")


    def build_journal_screen(self):
        self.journal_frame = tk.Frame(self.root, bg="#fdf6f0")
        self.journal_frame.pack(fill="both", expand=True)

        tk.Label(self.journal_frame, text="Journal-Based Mental Health Detection",
                 font=("Arial", 20, "bold"), bg="#fdf6f0", fg=self.TEXT_COLOR).pack(pady=15)

        tk.Label(self.journal_frame, text="How are you feeling today? Write freely:",
                 font=("Arial", 12), bg="#fdf6f0", fg=self.TEXT_COLOR).pack(anchor="w", padx=20)

        self.text_entry = tk.Text(self.journal_frame, height=10, width=85, font=("Arial", 11), bg="#fff")
        self.text_entry.pack(padx=20, pady=10)

        submit_btn = tk.Button(self.journal_frame, text="Analyze Journal", command=self.submit_journal,
                               bg=self.PASTEL_PURPLE, fg=self.TEXT_COLOR, font=("Arial", 12))
        submit_btn.pack(pady=15)

        self.result_label = tk.Label(self.journal_frame, text="", font=("Arial", 14), bg="#fdf6f0", fg=self.TEXT_COLOR)
        self.result_label.pack(pady=10)

        self.breathing_btn = tk.Button(self.journal_frame, text="Start Breathing Exercise",
                                       command=self.start_breathing_exercise,
                                       bg="#a8e6cf", fg=self.TEXT_COLOR, font=("Arial", 12))

        back_btn = tk.Button(self.journal_frame, text="← Back to Dashboard", command=self.show_dashboard,
                             font=("Arial", 10), bg=self.PASTEL_YELLOW)

        back_btn.pack(pady=5)
        self.breathing_btn.pack(pady=10)
        self.breathing_btn.pack_forget()

    def submit_journal(self):
        user_text = self.text_entry.get("1.0", tk.END).strip()
        if not user_text:
            messagebox.showwarning("Empty Input", "Please write something in the journal.")
            return

        try:
            predicted_status, fuzzy = self.predictor.predict(user_text)

            self.result_label.config(
                text=f"Predicted Status: {predicted_status}\nEmotion (Journal-based): {fuzzy}"
            )
            if predicted_status.lower() in ["stress", "anxiety", "depression","suicidal"]:
                self.breathing_btn.pack()
               

            else:
                self.breathing_btn.pack_forget()

            self.text_entry.delete("1.0", tk.END)
        except Exception as e:
            messagebox.showerror("Prediction Error", str(e))

    # --- Breathing Exercise Section ---
    def start_breathing_exercise(self):
        self.clear_frames()
    
        self.breathing_frame = tk.Frame(self.root, bg="#e4f9f5")
        self.breathing_frame.pack(fill="both", expand=True)
    
        title = tk.Label(self.breathing_frame, text="🧘 Breathing Exercise", font=("Arial", 24, "bold"), bg="#e4f9f5")
        title.pack(pady=20)
    
        self.instruction_label = tk.Label(self.breathing_frame, text="", font=("Arial", 20), bg="#e4f9f5")
        self.instruction_label.pack(pady=10)
    
        self.timer_label = tk.Label(self.breathing_frame, text="", font=("Arial", 16), bg="#e4f9f5")
        self.timer_label.pack(pady=5)
    
        self.progress = ttk.Progressbar(self.breathing_frame, orient="horizontal", length=400, mode="determinate")
        self.progress.pack(pady=10)
    
        self.back_button = tk.Button(self.breathing_frame, text="← Back to Dashboard",
                                     command=self.show_dashboard, bg=self.PASTEL_YELLOW, font=("Arial", 10))
        self.back_button.pack(pady=10)
        self.back_button.pack_forget()

         #back_btn = tk.Button(self.journal_frame, text="← Back to Dashboard", command=self.show_dashboard,
                #             font=("Arial", 10), bg=self.PASTEL_YELLOW)

               # back_btn.pack(pady=5)
        # Start phases
        self.breathing_phases = [("Breathe In", 4), ("Hold", 3), ("Breathe Out", 4)]
        self.current_phase = 0
        self.start_phase()
    def return_to_dashboard():
        self.breathing_frame.pack_forget()  # Hide the breathing frame
        self.breathing_frame.destroy()      # Fully destroy the frame
        self.show_dashboard()    
    
    def start_phase(self):
        if self.current_phase < len(self.breathing_phases):
            phase, duration = self.breathing_phases[self.current_phase]
            self.instruction_label.config(text=phase)
            self.start_timer(duration, self.start_phase)
            self.current_phase += 1
        else:
            self.instruction_label.config(text="✅ Exercise Complete!")
            self.timer_label.config(text="")
            self.progress["value"] = 100
            self.back_button = tk.Button(self.breathing_frame, text="← Back to Dashboard",
                                     command=self.show_dashboard, bg=self.PASTEL_YELLOW, font=("Arial", 10))
            self.back_button.pack(pady=10)
            
           # Show dashboard properly
            
            self.root.after(4000, self.return_to_dashboard)

    
    def start_timer(self, seconds, callback):
        self.remaining_time = seconds
        self.progress["value"] = 0
        self.progress["maximum"] = seconds
    
        def countdown():
            if self.remaining_time >= 0:
                self.timer_label.config(text=f"{self.remaining_time} seconds")
                self.progress["value"] = seconds - self.remaining_time
                self.remaining_time -= 1
                self.root.after(1000, countdown)
            else:
                callback()

        countdown()

    def breathing_phase(self, phase, duration, next_phase_callback):
        self.instruction_label.config(text=phase)
        self.progress["value"] = 0
        steps = 100
        interval = duration // steps

        def animate(i=0):
            if i <= steps:
                self.progress["value"] = i
                self.root.after(interval, animate, i + 1)
            else:
                next_phase_callback()

        animate()


if __name__ == "__main__":
    root = tk.Tk()
    app = MentalHealthApp(root)
    root.mainloop()